# Chapter 08: Optimisation and Model Selection
**Part II — Machine Learning Core**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

Gradient descent variants, learning rate scheduling, regularisation, and hyperparameter optimisation.

## Learning Objectives
See Chapter 8 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Text classification with Multinomial Naive Bayes


In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# Text classification with Multinomial Naive Bayes
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
    ('clf',   MultinomialNB(alpha=0.1))
])

pipeline.fit(X_train_text, y_train)
print(pipeline.score(X_test_text, y_test))

## SGD with momentum


In [ ]:
import torch
import torch.optim as optim

# SGD with momentum
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam
optimizer_adam = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999))

# AdamW with weight decay (recommended for transformers)
optimizer_adamw = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Training loop
for X_batch, y_batch in dataloader:
    optimizer_adamw.zero_grad()
    y_pred = model(X_batch)
    loss = loss_fn(y_pred, y_batch)
    loss.backward()
    optimizer_adamw.step()

## Visualising optimiser trajectories on a 2D loss surface


In [ ]:
# Visualising optimiser trajectories on a 2D loss surface
import numpy as np
import matplotlib.pyplot as plt

def loss(w1, w2): return (w1**2) / 10 + (w2**2)   # elongated bowl
def grad(w1, w2): return np.array([2*w1/10, 2*w2])

def sgd(start, lr=0.5, steps=40):
    path = [start.copy()]
    w = start.copy()
    for _ in range(steps):
        g = grad(w[0], w[1])
        w -= lr * g
        path.append(w.copy())
    return np.array(path)

def momentum_sgd(start, lr=0.5, beta=0.9, steps=40):
    path = [start.copy()]
    w = start.copy(); v = np.zeros(2)
    for _ in range(steps):
        g = grad(w[0], w[1])
        v = beta * v + (1-beta) * g
        w -= lr * v
        path.append(w.copy())
    return np.array(path)

def adam_opt(start, lr=0.5, beta1=0.9, beta2=0.999, eps=1e-8, steps=40):
    path = [start.copy()]
    w = start.copy(); m = np.zeros(2); v = np.zeros(2); t = 0
    for _ in range(steps):
        t += 1
        g = grad(w[0], w[1])
        m = beta1*m + (1-beta1)*g
        v = beta2*v + (1-beta2)*g**2
        mh = m / (1 - beta1**t); vh = v / (1 - beta2**t)
        w -= lr * mh / (np.sqrt(vh) + eps)
        path.append(w.copy())
    return np.array(path)

w1 = np.linspace(-6, 6, 200); w2 = np.linspace(-3, 3, 200)
W1, W2 = np.meshgrid(w1, w2)
Z = loss(W1, W2)

start = np.array([5.5, 2.5])
p_sgd  = sgd(start.copy())
p_mom  = momentum_sgd(start.copy())
p_adam = adam_opt(start.copy())

fig, ax = plt.subplots(figsize=(12, 6))
ax.contour(W1, W2, Z, levels=20, colors='gray', alpha=0.4, linewidths=0.7)
ax.contourf(W1, W2, Z, levels=20, cmap='Blues', alpha=0.2)
ax.plot(*p_sgd.T,  'r-o', markersize=4, lw=1.5, label='SGD', alpha=0.8)
ax.plot(*p_mom.T,  'g-s', markersize=4, lw=1.5, label='SGD + Momentum', alpha=0.8)
ax.plot(*p_adam.T, 'b-^', markersize=4, lw=1.5, label='Adam', alpha=0.8)
ax.plot(0, 0, 'k*', markersize=15, label='Optimum')
ax.plot(*start, 'ko', markersize=8)
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Optimiser trajectories on an elongated loss surface', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ch08_optimisers.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 📚 Review Questions

See Chapter 8 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 8 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*